# 제1장: AI 거버넌스 개념과 필요성

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch01.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

AI 거버넌스란 AI 시스템의 개발·배포·운영 전 과정에서 **책임성·투명성·공정성**을 확보하는 정책·프로세스·기술의 체계입니다.
이 장에서는 거버넌스 부재가 실제로 어떤 피해를 초래하는지 살펴보고,
`GovernanceHealthCheck` 도구로 프로젝트의 준수 상태를 즉시 자동 진단해 봅니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

## GovernanceHealthCheck — 거버넌스 자동 진단 도구

AI 프로젝트의 거버넌스 준수 상태를 **8개 항목**으로 자동 점검합니다.

| 영역 | 점검 항목 | 심각도 |
|------|----------|--------|
| 정책 | 모델 카드, 윤리 검토서, 위험 평가서, 데이터 거버넌스 정책 | CRITICAL |
| 프로세스 | CI/CD 파이프라인, 테스트 코드 디렉토리 | CRITICAL |
| 기술 | 모델 버전 관리 도구, 모니터링 설정 | WARNING |

아래 셀에서 `Severity`, `CheckResult`, `GovernanceReport`, `GovernanceHealthCheck` 클래스를 정의합니다.

In [ ]:
import os
import json
from dataclasses import dataclass, field
from typing import List, Dict
from enum import Enum

class Severity(Enum):
    CRITICAL = "CRITICAL"  # 즉시 조치 필요
    WARNING = "WARNING"    # 개선 권고
    INFO = "INFO"          # 참고 사항

@dataclass
class CheckResult:
    category: str          # 거버넌스 영역
    check_name: str        # 점검 항목명
    passed: bool           # 통과 여부
    severity: Severity     # 심각도
    message: str           # 상세 메시지
    recommendation: str    # 개선 권고사항

@dataclass
class GovernanceReport:
    project_name: str
    results: List[CheckResult] = field(default_factory=list)

    @property
    def score(self) -> float:
        if not self.results:
            return 0.0
        passed = sum(1 for r in self.results if r.passed)
        return round(passed / len(self.results) * 100, 1)

    @property
    def critical_issues(self) -> List[CheckResult]:
        return [r for r in self.results
                if not r.passed and r.severity == Severity.CRITICAL]

    def summary(self) -> Dict:
        return {
            "project": self.project_name,
            "total_checks": len(self.results),
            "passed": sum(1 for r in self.results if r.passed),
            "failed": sum(1 for r in self.results if not r.passed),
            "score": self.score,
            "critical_count": len(self.critical_issues),
        }

class GovernanceHealthCheck:
    """AI 프로젝트 거버넌스 Health Check 수행 클래스"""

    def __init__(self, project_root: str):
        self.project_root = project_root
        self.report = GovernanceReport(
            project_name=os.path.basename(project_root)
        )

    # --- 정책(Policy) 영역 점검 ---
    def check_policy_documents(self):
        """필수 정책 문서 존재 여부 점검"""
        required_docs = {
            "MODEL_CARD.md": "모델 카드",
            "ETHICS_REVIEW.md": "윤리 검토서",
            "RISK_ASSESSMENT.md": "위험 평가서",
            "DATA_GOVERNANCE.md": "데이터 거버넌스 정책",
        }
        for filename, doc_name in required_docs.items():
            exists = os.path.exists(
                os.path.join(self.project_root, "docs", filename)
            )
            self.report.results.append(CheckResult(
                category="정책",
                check_name=f"{doc_name} 존재 여부",
                passed=exists,
                severity=Severity.CRITICAL,
                message=f"{doc_name} {'존재' if exists else '미존재'}",
                recommendation="" if exists else
                    f"docs/{filename} 작성 필요"
            ))

    # --- 프로세스(Process) 영역 점검 ---
    def check_cicd_pipeline(self):
        """CI/CD 파이프라인 설정 존재 여부 점검"""
        ci_files = [
            ".github/workflows", ".gitlab-ci.yml",
            "Jenkinsfile", ".circleci/config.yml"
        ]
        found = any(
            os.path.exists(os.path.join(self.project_root, f))
            for f in ci_files
        )
        self.report.results.append(CheckResult(
            category="프로세스",
            check_name="CI/CD 파이프라인 설정",
            passed=found,
            severity=Severity.CRITICAL,
            message="CI/CD 파이프라인 " +
                    ("설정됨" if found else "미설정"),
            recommendation="" if found else
                "자동화된 빌드/테스트/배포 파이프라인 구축 필요"
        ))

    def check_test_coverage(self):
        """테스트 코드 존재 여부 점검"""
        test_dirs = ["tests", "test", "spec"]
        found = any(
            os.path.isdir(os.path.join(self.project_root, d))
            for d in test_dirs
        )
        self.report.results.append(CheckResult(
            category="프로세스",
            check_name="테스트 코드 디렉토리",
            passed=found,
            severity=Severity.CRITICAL,
            message="테스트 디렉토리 " +
                    ("존재" if found else "미존재"),
            recommendation="" if found else
                "단위/통합/공정성 테스트 작성 필요"
        ))

    # --- 기술(Technology) 영역 점검 ---
    def check_model_versioning(self):
        """모델 버전 관리 체계 점검"""
        version_indicators = [
            "mlflow", "wandb", "dvc", "mlflow.log_model"
        ]
        # requirements.txt 또는 pyproject.toml 검사
        req_file = os.path.join(
            self.project_root, "requirements.txt"
        )
        found = False
        if os.path.exists(req_file):
            with open(req_file, "r") as f:
                content = f.read().lower()
                found = any(v in content
                           for v in version_indicators)
        self.report.results.append(CheckResult(
            category="기술",
            check_name="모델 버전 관리 도구",
            passed=found,
            severity=Severity.WARNING,
            message="모델 버전 관리 도구 " +
                    ("감지됨" if found else "미감지"),
            recommendation="" if found else
                "MLflow, DVC 등 모델 레지스트리 도입 권장"
        ))

    def check_monitoring_config(self):
        """모니터링 설정 존재 여부 점검"""
        monitoring_indicators = [
            "evidently", "whylabs", "arize",
            "prometheus", "grafana", "monitoring"
        ]
        found = any(
            os.path.exists(
                os.path.join(self.project_root, f"monitoring")
            )
            for _ in [1]
        )
        self.report.results.append(CheckResult(
            category="기술",
            check_name="모니터링 설정",
            passed=found,
            severity=Severity.WARNING,
            message="모니터링 설정 " +
                    ("존재" if found else "미존재"),
            recommendation="" if found else
                "모델 성능/드리프트 모니터링 체계 구축 권장"
        ))

    # --- 전체 점검 실행 ---
    def run_all_checks(self) -> GovernanceReport:
        """모든 거버넌스 점검 항목 실행"""
        self.check_policy_documents()
        self.check_cicd_pipeline()
        self.check_test_coverage()
        self.check_model_versioning()
        self.check_monitoring_config()
        return self.report

    def print_report(self):
        """거버넌스 점검 결과 출력"""
        report = self.run_all_checks()
        summary = report.summary()

        print("=" * 60)
        print(f"AI Governance Health Check: {summary['project']}")
        print(f"Score: {summary['score']}% "
              f"({summary['passed']}/{summary['total_checks']})")
        print("=" * 60)

        for result in report.results:
            status = "PASS" if result.passed else "FAIL"
            icon = "[+]" if result.passed else "[-]"
            print(f"  {icon} [{result.category}] "
                  f"{result.check_name}: {status}")
            if not result.passed:
                print(f"      -> {result.recommendation}")

        if report.critical_issues:
            print(f"\n!! {len(report.critical_issues)} "
                  f"CRITICAL issues found !!")

## 데모: 거버넌스 미적용 vs. 적용 프로젝트 비교

두 가지 시나리오를 임시 디렉토리로 시뮬레이션합니다.

- **DEMO 1 — 미적용 프로젝트**: 빈 디렉토리 → 거버넌스 문서·CI·테스트 없음 → **0/8점**, CRITICAL 이슈 6개
- **DEMO 2 — 적용 프로젝트**: 필수 파일을 자동 생성 → 8개 항목 통과 → **100점**

> 실무에서는 자신의 AI 프로젝트 루트 경로를 `GovernanceHealthCheck("경로")` 에 넣어 즉시 진단할 수 있습니다.

In [ ]:
import tempfile, os

bad_project = tempfile.mkdtemp(prefix="ungoverned_ai_")
print("[DEMO 1] 거버넌스 미적용 프로젝트")
print(f"  경로: {bad_project}")
checker_bad = GovernanceHealthCheck(bad_project)
checker_bad.print_report()

# ── DEMO 2: 거버넌스 적용 프로젝트 (통과 사례) ───────────────────
good_project = tempfile.mkdtemp(prefix="governed_ai_")

# 필수 정책 문서 생성
os.makedirs(os.path.join(good_project, "docs"), exist_ok=True)
for doc in ["MODEL_CARD.md", "ETHICS_REVIEW.md",
            "RISK_ASSESSMENT.md", "DATA_GOVERNANCE.md"]:
    open(os.path.join(good_project, "docs", doc), "w").write(f"# {doc}")

# CI/CD 파이프라인 설정
os.makedirs(os.path.join(good_project, ".github", "workflows"), exist_ok=True)
open(os.path.join(good_project, ".github", "workflows", "ci.yml"), "w").write("# CI")

# 테스트 디렉토리
os.makedirs(os.path.join(good_project, "tests"), exist_ok=True)

# 모델 버전 관리 도구 (requirements.txt에 mlflow)
open(os.path.join(good_project, "requirements.txt"), "w").write("mlflow>=2.0")

# 모니터링 디렉토리
os.makedirs(os.path.join(good_project, "monitoring"), exist_ok=True)

print("")
print("[DEMO 2] 거버넌스 적용 프로젝트")
print(f"  경로: {good_project}")
checker_good = GovernanceHealthCheck(good_project)
checker_good.print_report()

## 결과 해석

| 점수 | 위험 등급 | 의미 |
|------|---------|------|
| 85% 이상 | 🟢 LOW | 거버넌스 기반 갖춤, 정기 점검 유지 |
| 60~84% | 🟡 MEDIUM | 일부 항목 보완 필요, 2주 내 개선 계획 수립 권장 |
| 60% 미만 | 🔴 HIGH | 즉각 조치 필요, AI 기본법 §7 위반 위험 |

**CRITICAL 이슈**는 즉시 해결해야 합니다. `recommendation` 필드의 안내를 따라 누락 문서를 작성하고
CI/CD·테스트 체계를 구축하면 다음 실행 시 점수가 올라갑니다.

> 다음 장(ch02)에서는 NIST AI RMF와 EU AI Act 기준으로 더 정밀한 규제 준수 평가를 수행합니다.